# Import libraries

In [7]:
# Changes to all modules will automatically be applied when any cell runs. 
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
import pandas as pd
import numpy as np

from pytorch_tabular.models import CategoryEmbeddingModelConfig, TabNetModelConfig

from pathlib import Path
import sys

from sklearn.metrics import (
    make_scorer,
    f1_score,
)

sys.path.append(
    str(Path('..', 'utils_functionality', 'split_utils'))
)
sys.path.append(
    str(Path('..', 'utils_functionality', 'models'))
)

from split_tools import get_train_test
from modelling4_utils import (
    # _prepare_df,
    MLPipeline,
    StatsModelsEstimator,
    PytorchTabularEstimator,
    update_estimator_params,
    OptunaOptimizer,
    GridSearchOptimizer,
    RANDOM_STATE,
)

# from pytorch_lightning.loggers import MLFlowLogger
from pytorch_lightning.loggers import TensorBoardLogger

seed = RANDOM_STATE

In [9]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Settings

In [10]:
model_postfix = 'first_launch'

scoring_metrics = {"f1_macro": make_scorer(f1_score, average="macro"),}
step_scoring_average = "mean"
# n_trials = 1500 # checked with 15 and 100

save_model_and_metrics = False
metrics_file = "metrics_modelling4_6-neural_networks.xlsx"

## MultiLayer Perceptron (MLP)

In [11]:
estimator = PytorchTabularEstimator
target = "splashing"

add_smote = True
is_smotenc = False
smote_params = {
    # 'categorical_features': (
    #     'wettability',
    #     'inclination',
    # ),
}

estimator_params = {
    "model_class": CategoryEmbeddingModelConfig,
    "model_config_params": {
        'layers': '2',
        'activation': 'LeakyReLU',
        # 'activation_config': {'negative_slope': 0.01},
        'dropout': 0.2,
        'batch_norm_continuous_input': False, # We have normalized features
        'learning_rate': 1e-3,
    },
    # "data_config_params": {},
    "trainer_config_params": {
        'max_epochs': 200,
        'early_stopping': 'valid_loss',
        'early_stopping_patience': 10,
        'batch_size': 300,
        'accelerator': "cpu",  # вместо auto или mps
        # 'output_dir': Path("..", "results", "models_modelling4"), # take from ml_pipe
        # 'log_target': 'wandb',
        # 'logger_name': 'default',
        # 'checkpoints': 'f1_macro',
    },
    "optimizer_config_params": {
        'optimizer': 'Adam',
        'lr_scheduler': 'ReduceLROnPlateau',
        'lr_scheduler_params': {
            'patience': 5,
            'factor': 0.5,
        },
    },
}

ml_pipe = MLPipeline(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    model_postfix=model_postfix,
    add_smote=add_smote,
    is_smotenc=is_smotenc,
    smote_params=smote_params,
    metrics_file=metrics_file,
    cv_folds=3,
)

ml_pipe.run(
    save_model_and_metrics=save_model_and_metrics,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

2025-04-26 22:15:42,925 - {pytorch_tabular.tabular_model:146} - INFO - Experiment Tracking is turned off

Seed set to 42


...

`Trainer.fit` stopped: `max_epochs=200` reached.


2025-04-26 22:16:23,191 - {pytorch_tabular.tabular_model:689} - INFO - Training the model completed

2025-04-26 22:16:23,206 - {pytorch_tabular.tabular_model:1529} - INFO - Loading the best model

no summary in estimator class "PytorchTabularEstimator"


,0
target,splashing
model,CategoryEmbeddingModel_splashing_smote_first_l...
holdout_test_f1_macro,0.674654
holdout_test_accuracy_balanced,0.667824
holdout_test_roc_auc,0.820988
holdout_test_f1,0.796117
holdout_test_accuracy,0.72
cv_test_f1_macro_median,0.606349
cv_test_accuracy_balanced_median,0.606818
cv_test_roc_auc_median,0.700284


In [13]:
ml_pipe.pipe[-1].model.config

{'target': ['target'], 'continuous_cols': ['inclination', 'volume_fraction', 'sedimentation_Stk', 'particle_droplet_diameter_ratio', 'relative_roughness', 'K', 'wettability'], 'categorical_cols': [], 'date_columns': [], 'encode_date_columns': True, 'validation_split': 0.2, 'continuous_feature_transform': None, 'normalize_continuous_features': True, 'quantile_noise': 0, 'num_workers': 0, 'pin_memory': True, 'handle_unknown_categories': True, 'handle_missing_values': True, 'dataloader_kwargs': {}, 'task': 'classification', 'head': 'LinearHead', 'head_config': {'layers': ''}, 'embedding_dims': None, 'embedding_dropout': 0.0, 'batch_norm_continuous_input': True, 'learning_rate': 0.001, 'loss': 'CrossEntropyLoss', 'metrics': ['f1_score', 'accuracy'], 'metrics_prob_input': [False, False], 'metrics_params': [{}, {}], 'target_range': None, 'virtual_batch_size': None, 'seed': 42, '_module_src': 'models.category_embedding', '_model_name': 'CategoryEmbeddingModel', '_backbone_name': 'CategoryEmbe

In [ ]:
estimator = PytorchTabularEstimator
target = "no_fragmentation"

add_smote = True
is_smotenc = True
smote_params = {
    'categorical_features': (
        'wettability',
        'inclination',
    ),
}

estimator_params = {
    "model_class": CategoryEmbeddingModelConfig,
    "model_config_params": {
        'activation': 'LeakyReLU',
        'dropout': 0.2,
        'batch_norm_continuous_input': True, # We have normalized features
        'learning_rate': 1e-3,
    },
    # "data_config_params": {},
    "trainer_config_params": {
        'max_epochs': 200,
        'early_stopping': 'valid_loss',
        'early_stopping_patience': 10,
    },
    "optimizer_config_params": {
        'optimizer': 'Adam',
        'lr_scheduler': 'ReduceLROnPlateau',
        'lr_scheduler_params': {
            'patience': 5,
            'factor': 0.5,
        },
    },
}

ml_pipe = MLPipeline(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    model_postfix=model_postfix,
    add_smote=add_smote,
    is_smotenc=is_smotenc,
    smote_params=smote_params,
    metrics_file=metrics_file,
)

ml_pipe.run(
    save_model_and_metrics=save_model_and_metrics,
)